# Mangrove Segmentation — Kaggle GPU Edition v6

Converted from the exported Kaggle source into a runnable notebook.

**Note on cell order:** in the exported source, the dataset-diagnostic
snippets (folder listing, band inspection, input-structure walk) appeared
*before* the cell that defines `CONFIG`, which would raise `NameError:
CONFIG` on a fresh top-to-bottom run. They have been moved to **after**
the setup/definitions cell, where `CONFIG` already exists. Logic is
otherwise unchanged.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


In [ ]:
"""
================================================================
  MANGROVE SEGMENTATION — KAGGLE GPU EDITION v4 (RUNTIME FIXED)
  Sentinel-2 | T4/P100 GPU | mixed_float16 | Drone inference
================================================================

CRITICAL BUG FIXED IN THIS REVISION
─────────────────────────────────────
Error:  rasterio.errors.EnvError: No GDAL environment exists

Root cause (confirmed from contextlib frames in traceback):
  rasterio.open() is a @contextmanager that internally executes:
      with Env(**opts):          ← creates its OWN Env
          yield DatasetBase(...)

  rasterio's delenv() sets thread-local._env = None unconditionally —
  there is NO reference counting.  When rasterio.open().__exit__() fires,
  the inner Env.__exit__() calls delenv() → local._env = None.
  The outer explicit `with rasterio.Env():` then calls delenv() on a
  None env → EnvError: No GDAL environment exists.

  This only manifests in tf.data background threads because the global
  GDAL_ENV.__enter__() called in the main thread has zero effect on those
  threads (GDAL env is thread-local).

Two-line surgical fix:
  [1] Removed  global Env().__enter__() block.
      GDAL options moved to os.environ — process-wide, picked up by
      every rasterio.open() in every thread at GDAL initialisation time.

  [2] Removed  `with rasterio.Env():` from patch_generator.
      rasterio.open() is now the sole GDAL env manager per file/thread.
      No nesting → no double delenv() → no crash.

All three prior fixes retained:
  [Fix 1] generate_mask() with precomputed NDVI (band 6) + NDMI (band 8)
  [Fix 2] IoUMetric class-based metric  (Keras 3 callback compatibility)
  [Fix 3] steps_per_epoch + validation_steps in model.fit()
================================================================
"""

import os
import sys

# 1. FORCE PROCESS-WIDE GDAL ENV VARIABLES (Must happen FIRST)
os.environ["GDAL_CACHEMAX"] = "512"                  # 512 MB tile cache [cite: 11]
os.environ["GDAL_NUM_THREADS"] = "ALL_CPUS"          # Enable multi-threaded reading [cite: 11]
os.environ["CPL_DEBUG"] = "FALSE"                    # Mute verbose warnings [cite: 11]
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR" # Hyper-speed folder bypass [cite: 11]
os.environ["CPL_VSIL_USE_TEMP_FILE_FOR_RANDOM_WRITE"] = "NO" # [cite: 11]

# 2. NOW IMPORT DEPENDENCIES
import cv2
import numpy as np
import rasterio
import rasterio.windows
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

cv2.setNumThreads(os.cpu_count())

# CRITICAL SANITY CHECK: Double-check that no stray global Env blocks exist!
# Do NOT call rasterio.Env() anywhere down here. [cite: 10]


# ─────────────────────────────────────────────
# 1. GPU SETUP
# ─────────────────────────────────────────────
tf.keras.mixed_precision.set_global_policy("mixed_float16")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU     : {len(gpus)}× detected")
else:
    print("WARNING: No GPU detected — remove mixed_float16 for CPU runs")
print(f"Policy  : {tf.keras.mixed_precision.global_policy().name}")
print(f"TF      : {tf.__version__}")

strategy = tf.distribute.MirroredStrategy()
print(f"Replicas in sync: {strategy.num_replicas_in_sync}") 

# ─────────────────────────────────────────────
# 2. PATHS
# ─────────────────────────────────────────────
KAGGLE_INPUT  = "/kaggle/input/datasets/aayushraman07/sundarban/SUNDARBAN"
KAGGLE_OUTPUT = "/kaggle/working"
MODEL_PATH    = os.path.join(KAGGLE_OUTPUT, "mangrove_v6.keras")
STATS_PATH    = os.path.join(KAGGLE_OUTPUT, "mangrove_v6_norm_stats.npz")


# ─────────────────────────────────────────────
# 3. CONFIG
#    Band layout in TIF (0-indexed):
#      [0]=B2  [1]=B3  [2]=B4  [3]=B8  [4]=B11
#      [5]=SCL (excluded — categorical)
#      [6]=NDVI  [7]=NDWI  [8]=NDMI
#      [9]=label (ALL ZEROS — unusable)
#
#    Global norm stats (confirmed from full-scene scan):
#      B2≈1547  B3≈1741  B4≈1582  B8≈2367  B11≈1840  (DN ×10000)
#      NDVI≈0.138  NDWI≈-0.091  NDMI≈0.106
# ─────────────────────────────────────────────
CONFIG = {
    "folder":            KAGGLE_INPUT,
    "patch_size":        512,
    "stride":            512,
    "window_size":       2048,
    "batch_size":        16,
    "max_train_patches": 2500,
    "max_val_patches":   500,
    "val_split":         0.2,
    "epochs":            50,

    # 8 input channels: B2, B3, B4, B8, B11, NDVI, NDWI, NDMI
    "input_bands":       [0, 1, 2, 3, 4, 6, 7, 8],

    # Precomputed index bands already stored in TIF — no recomputation needed
    "ndvi_band":         6,
    "ndmi_band":         8,

    # NDVI/NDMI thresholds tuned for Sundarbans
    # Scene-wide NDVI mean ≈ 0.138 (water pulls it down)
    # Mangrove canopy specifically: NDVI 0.4–0.8
    "ndvi_low":          0.30,
    "ndvi_high":         0.85,
    "ndmi_low":          0.05,

    "min_mask_pixels":   50,
    "min_blob_area":     100,
}

N_CHANNELS = len(CONFIG["input_bands"])   # 8

print(f"\nInput bands  : {CONFIG['input_bands']}  →  B2,B3,B4,B8,B11,NDVI,NDWI,NDMI")
print(f"N_CHANNELS   : {N_CHANNELS}")
print(f"Masking      : NDVI[{CONFIG['ndvi_low']},{CONFIG['ndvi_high']}] + NDMI>{CONFIG['ndmi_low']}")
print(f"Patch size   : {CONFIG['patch_size']}px  |  Stride: {CONFIG['stride']}px  |  Batch: {CONFIG['batch_size']}")
print(f"Train patches: {CONFIG['max_train_patches']}  |  Val: {CONFIG['max_val_patches']}")


# ─────────────────────────────────────────────
# 4. MASK GENERATION
#    Uses precomputed NDVI (band 6) + NDMI (band 8) from TIF.
#    Label band [9] is all zeros — abandoned.
# ─────────────────────────────────────────────
def generate_mask(all_bands_hwc, config):
    """
    Generate mangrove binary mask from precomputed NDVI and NDMI.

    Parameters
    ----------
    all_bands_hwc : np.ndarray  (H, W, 10)  float32 — full window
    config        : dict

    Returns
    -------
    mask : np.ndarray  (H, W)  float32  in {0.0, 1.0}
    """
    ndvi = all_bands_hwc[:, :, config["ndvi_band"]].astype(np.float32)
    ndmi = all_bands_hwc[:, :, config["ndmi_band"]].astype(np.float32)

    # Gaussian blur before threshold — reduces salt-and-pepper noise
    ndvi_blur = cv2.GaussianBlur(ndvi, (5, 5), 0)
    ndmi_blur = cv2.GaussianBlur(ndmi, (5, 5), 0)

    # Combined NDVI range gate + NDMI moisture gate
    mask = (
        (ndvi_blur > config["ndvi_low"])  &
        (ndvi_blur < config["ndvi_high"]) &
        (ndmi_blur > config["ndmi_low"])
    ).astype(np.uint8) * 255

    # Morphological cleanup — open removes noise, close fills gaps
    kernel = np.ones((5, 5), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Remove blobs smaller than min_blob_area pixels
    num_labels, labels_cc, stats, _ = cv2.connectedComponentsWithStats(mask)
    clean = np.zeros_like(mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] > config["min_blob_area"]:
            clean[labels_cc == i] = 255

    return (clean / 255.0).astype(np.float32)


# ─────────────────────────────────────────────
# 5. DIAGNOSTICS
# ─────────────────────────────────────────────
def diagnose_dataset(config):
    print("\n=== DATASET DIAGNOSTICS ===")
    folder = config["folder"]
    if not os.path.exists(folder):
        print(f"Folder not found: {folder}")
        return

    tif_files = sorted([f for f in os.listdir(folder) if f.lower().endswith(".tif")])
    print(f"TIF files : {len(tif_files)}")

    for fname in tif_files[:4]:
        path = os.path.join(folder, fname)
        with rasterio.open(path) as src:
            h, w   = src.height, src.width
            ws     = min(2048, h, w)
            cx     = max(0, w // 2 - ws // 2)
            cy     = max(0, h // 2 - ws // 2)
            window = rasterio.windows.Window(cx, cy, ws, ws)
            chunk  = src.read(window=window).astype(np.float32)   # (10, H, W)
        chunk_hw    = np.transpose(chunk, (1, 2, 0))               # (H, W, 10)
        mask_sample = generate_mask(chunk_hw, config)

        cov = 100 * mask_sample.mean()
        print(f"\n  {fname}")
        print(f"    Size     : {w}×{h}px")
        print(f"    NDVI     : mean={chunk_hw[:,:,6].mean():.3f}  "
              f"range=[{chunk_hw[:,:,6].min():.2f}, {chunk_hw[:,:,6].max():.2f}]")
        print(f"    NDMI     : mean={chunk_hw[:,:,8].mean():.3f}  "
              f"range=[{chunk_hw[:,:,8].min():.2f}, {chunk_hw[:,:,8].max():.2f}]")
        print(f"    Mangrove : {cov:.1f}% of sampled 2048px window")

    print("\n=== END DIAGNOSTICS ===\n")


# ─────────────────────────────────────────────
# 6. FILE-BASED TRAIN/VAL SPLIT
# ─────────────────────────────────────────────
def get_file_split(config):
    files = sorted([
        os.path.join(config["folder"], f)
        for f in os.listdir(config["folder"]) if f.endswith(".tif")
    ])
    if not files:
        raise ValueError(f"No .tif files in: {config['folder']}")
    if len(files) == 1:
        print("WARNING: Only 1 TIF — using it for both train and val.")
        return files, files

    train_files, val_files = train_test_split(
        files, test_size=config["val_split"], random_state=42
    )
    print(f"Train files : {len(train_files)} | Val files : {len(val_files)}")
    return train_files, val_files


# ─────────────────────────────────────────────
# 7. GLOBAL NORMALIZATION STATS
#    Computed once, cached as .npz; reloaded on subsequent runs.
# ─────────────────────────────────────────────
def compute_global_stats(config):
    print("Computing global normalization stats (windowed pass)...")
    running_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    running_sq  = np.zeros(N_CHANNELS, dtype=np.float64)
    pixel_count = 0
    ws          = config["window_size"]
    rasterio_bands = [b + 1 for b in config["input_bands"]]   # 1-indexed for rasterio

    tif_files = [f for f in os.listdir(config["folder"]) if f.endswith(".tif")]
    print(f"  Scanning {len(tif_files)} file(s) in {ws}px windows...")

    for fname in tif_files:
        path = os.path.join(config["folder"], fname)
        with rasterio.open(path) as src:
            full_h, full_w = src.height, src.width
            for row_off in range(0, full_h, ws):
                for col_off in range(0, full_w, ws):
                    win_h = min(ws, full_h - row_off)
                    win_w = min(ws, full_w - col_off)
                    if win_h < 1 or win_w < 1:
                        continue
                    window = rasterio.windows.Window(col_off, row_off, win_w, win_h)
                    chunk  = src.read(rasterio_bands, window=window).astype(np.float32)
                    chunk  = np.transpose(chunk, (1, 2, 0))   # (H, W, 8)
                    flat   = chunk.reshape(-1, N_CHANNELS)
                    running_sum += flat.sum(axis=0)
                    running_sq  += (flat ** 2).sum(axis=0)
                    pixel_count += flat.shape[0]

    mean = (running_sum / pixel_count).astype(np.float32)
    std  = (np.sqrt(np.maximum(running_sq / pixel_count - mean ** 2, 0)) + 1e-6).astype(np.float32)
    np.savez(STATS_PATH, mean=mean, std=std)

    band_names = ["B2", "B3", "B4", "B8", "B11", "NDVI", "NDWI", "NDMI"]
    for name, m, s in zip(band_names, mean, std):
        print(f"  {name:6s}: mean={m:.4f}  std={s:.4f}")
    print(f"  Saved: {STATS_PATH}")
    return mean, std


def load_or_compute_stats(config):
    if os.path.exists(STATS_PATH):
        d = np.load(STATS_PATH)
        print("Loaded cached normalization stats.")
        return d["mean"].astype(np.float32), d["std"].astype(np.float32)
    return compute_global_stats(config)


# ─────────────────────────────────────────────
# 8. STREAMING PATCH GENERATOR

# ══ THREADING CONTRACT ══════════════════════

# ════════════════════════════════════════════

import gc

def patch_generator(file_list, mean, std, config, augment=False, max_patches=99999):
    patch_count = 0
    ps = config["patch_size"]
    ws = config["window_size"]
    stride = config["stride"]
    min_mask_pixels = config.get("min_mask_pixels", 0)

    for path in file_list:
        print(f" File: {os.path.basename(path)}")
        windows_with_mask = 0

        # Manually open without the 'with' context manager to prevent the GDAL thread-teardown crash
        src = rasterio.open(path)
        try:
            full_h, full_w = src.height, src.width
            
            for row_off in range(0, full_h - ps + 1, ws):
                for col_off in range(0, full_w - ps + 1, ws):
                    if patch_count >= max_patches:
                        return
                        
                    win_h = min(ws, full_h - row_off)
                    win_w = min(ws, full_w - col_off)
                    if win_h < ps or win_w < ps:
                        continue
                        
                    window = rasterio.windows.Window(col_off, row_off, win_w, win_h)
                    all_bands = src.read(window=window).astype(np.float32)  # (10, H, W)
                    all_bands = np.transpose(all_bands, (1, 2, 0))          # (H, W, 10)
                    
                    mask = generate_mask(all_bands, config)
                    if np.count_nonzero(mask) == 0:
                        del all_bands
                        continue
                        
                    x_chunk = (all_bands[:, :, config["input_bands"]] - mean) / std
                    x_chunk = x_chunk.astype(np.float32)
                    windows_with_mask += 1
                    
                    del all_bands

                    # On-the-fly streaming engine keeping RAM usage completely flat
                    h, w = mask.shape
                    for i in range(0, h - ps + 1, stride):
                        for j in range(0, w - ps + 1, stride):
                            if patch_count >= max_patches:
                                return
                                
                            y_patch = mask[i:i+ps, j:j+ps]
                            if np.sum(y_patch) < min_mask_pixels:
                                continue
                                
                            x_patch = x_chunk[i:i+ps, j:j+ps, :]
                            y_patch = np.expand_dims(y_patch, -1)
                            
                            # ========================================================
                            # OPTIMIZED YIELD STEP:
                            # Stream out only the original raw patch data.
                            # Completely cuts out the manual, RAM-bloating 8x loops.
                            # ========================================================
                            yield x_patch.astype(np.float32), y_patch.astype(np.float32)
                            patch_count += 1
                    
                    del x_chunk, mask
        finally:
            # Safely close the raw handle link directly
            src.close()
                    
        gc.collect()
        print(f" → {patch_count} patches total generated | {windows_with_mask} windows had mangrove signal")
# ─────────────────────────────────────────────
# 9. tf.data PIPELINE
# ─────────────────────────────────────────────

  # ─────────────────────────────────────────────
# AUGMENTATION (applied inside tf.data pipeline)
# ─────────────────────────────────────────────
def tf_augment(x, y):
    """
    Real-time random flip/rotation augmentation, applied per-sample
    inside the tf.data graph. No extra RAM — operates on one tensor
    at a time, no pre-multiplied 8x copies.
    """
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_left_right(x)
        y = tf.image.flip_left_right(y)

    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_up_down(x)
        y = tf.image.flip_up_down(y)

    rotations = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    x = tf.image.rot90(x, k=rotations)
    y = tf.image.rot90(y, k=rotations)

    return x, y


# ─────────────────────────────────────────────
# tf.data PIPELINE
# ─────────────────────────────────────────────
def make_dataset(file_list, mean, std, config, augment=False, shuffle=False,
                 max_patches=99999, cache_path=None):
    ps = config["patch_size"]
    
    # [Optimization] Keep labels float32 but prepare to match pipeline precision
    output_sig = (
        tf.TensorSpec(shape=(ps, ps, N_CHANNELS), dtype=tf.float32),
        tf.TensorSpec(shape=(ps, ps, 1),          dtype=tf.float32),
    )

    # Wrap the generator in a lambda factory style constructor
    ds = tf.data.Dataset.from_generator(
        lambda: patch_generator(file_list, mean, std, config, max_patches=max_patches),
        output_signature=output_sig
    )

    # 1. Cache BEFORE shuffle/repeat to ensure the raw generator is only read ONCE
    if cache_path:
        ds = ds.cache(filename=cache_path)

    # 2. [Optimization] Drop shuffle buffer from 500 to 100. 
    # Saves ~3.5GB of RAM overhead, boosting pipeline structural tracking.
    if shuffle:
        ds = ds.shuffle(buffer_size=100, seed=42)

    ds = ds.repeat()

    # 3. Parallelize the data augmentations using CPU threads
    if augment:
        ds = ds.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)

    # 4. Batch and eagerly prefetch arrays directly to the GPU 
    return ds.batch(config["batch_size"], drop_remainder=True).prefetch(tf.data.AUTOTUNE)
# ─────────────────────────────────────────────
# 10. LOSS FUNCTIONS
# ─────────────────────────────────────────────
def dice_loss(y_true, y_pred):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    yt     = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    yp     = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])
    inter  = tf.reduce_sum(yt * yp, axis=1)
    denom  = tf.reduce_sum(yt, axis=1) + tf.reduce_sum(yp, axis=1)
    return tf.reduce_mean(1.0 - (2.0 * inter + smooth) / (denom + smooth))


def bce_dice_loss(y_true, y_pred):
    y_true  = tf.cast(y_true, tf.float32)
    y_pred  = tf.cast(y_pred, tf.float32)
    yt_flat = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    yp_flat = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])
    bce     = tf.reduce_mean(tf.keras.losses.binary_crossentropy(yt_flat, yp_flat))
    return bce + dice_loss(y_true, y_pred)


# ─────────────────────────────────────────────
# 11. IoU METRIC (class-based — required by Keras 3)
#     Plain functions are not registered as named metrics in Keras 3
#     (TF 2.19+), so callbacks cannot monitor "val_iou_metric".
#     A class-based Metric is tracked correctly.
# ─────────────────────────────────────────────
class IoUMetric(tf.keras.metrics.Metric):
    def __init__(self, threshold=0.5, name="iou_metric", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.iou_sum   = self.add_weight(name="iou_sum", initializer="zeros")
        self.count     = self.add_weight(name="count",   initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true > 0.5,            tf.float32)
        yt     = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
        yp     = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])
        inter  = tf.reduce_sum(yt * yp, axis=1)
        union  = tf.reduce_sum(yt, axis=1) + tf.reduce_sum(yp, axis=1) - inter
        iou    = tf.reduce_mean((inter + 1e-6) / (union + 1e-6))
        self.iou_sum.assign_add(iou)
        self.count.assign_add(1.0)

    def result(self):
        return self.iou_sum / (self.count + 1e-6)

    def reset_state(self):
        self.iou_sum.assign(0.0)
        self.count.assign(0.0)


# ─────────────────────────────────────────────
# 12. MODEL — Residual U-Net (8-channel input)
# ─────────────────────────────────────────────
def conv_block(x, filters, dropout=0.0):
    shortcut = layers.Conv2D(filters, 1, padding="same")(x)
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    if dropout > 0.0:
        x = layers.Dropout(dropout)(x)
    x = layers.add([x, shortcut])
    x = layers.Activation("relu")(x)
    return x


def upsample_block(x, filters):
    # dtype="float32" bypasses the XLA gradient casting bug with mixed_float16
    x = layers.UpSampling2D(size=(2, 2), interpolation="bilinear", dtype="float32")(x)
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x


def build_unet(input_shape):
    inputs = layers.Input(input_shape)                                                     # (512,512,8)

    c1 = conv_block(inputs, 32);  p1 = layers.MaxPooling2D()(c1)                          # 256
    c2 = conv_block(p1,     64);  p2 = layers.MaxPooling2D()(c2)                          # 128
    c3 = conv_block(p2,    128);  p3 = layers.MaxPooling2D()(c3)                          # 64
    c4 = conv_block(p3,    256);  p4 = layers.MaxPooling2D()(c4)                          # 32
    b  = conv_block(p4,    512, dropout=0.3)                                               # bottleneck

    u1 = upsample_block(b,   256); u1 = layers.concatenate([u1, c4]); c5 = conv_block(u1, 256)
    u2 = upsample_block(c5,  128); u2 = layers.concatenate([u2, c3]); c6 = conv_block(u2, 128)
    u3 = upsample_block(c6,   64); u3 = layers.concatenate([u3, c2]); c7 = conv_block(u3,  64)
    u4 = upsample_block(c7,   32); u4 = layers.concatenate([u4, c1]); c8 = conv_block(u4,  32)

    # float32 output — required with mixed_float16 global policy
    outputs = layers.Activation("sigmoid", dtype="float32")(layers.Conv2D(1, 1)(c8))
    return models.Model(inputs, outputs, name="MangroveUNet_v6")


# ─────────────────────────────────────────────
# 13. VISUALIZATION
# ─────────────────────────────────────────────
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history.get("loss",           []), label="Train")
    ax1.plot(history.history.get("val_loss",       []), label="Val")
    ax1.set_title("Loss"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(history.history.get("iou_metric",     []), label="Train")
    ax2.plot(history.history.get("val_iou_metric", []), label="Val")
    ax2.set_title("IoU"); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    out = os.path.join(KAGGLE_OUTPUT, "training_history.png")
    plt.savefig(out, dpi=150); plt.show()
    print(f"Saved: {out}")


def visualize_predictions(model, val_ds, n_samples=4):
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 3 * n_samples))
    for ax, t in zip(axes[0], ["RGB (B4,B3,B2)", "NDVI mask (label)", "Prediction"]):
        ax.set_title(t, fontsize=10)

    count = 0
    for x_batch, y_batch in val_ds.take(3):
        preds = model.predict(x_batch, verbose=0)
        for i in range(min(len(x_batch), n_samples - count)):
            x   = x_batch[i].numpy()
            y   = y_batch[i].numpy()[:, :, 0]
            p   = preds[i][:, :, 0]
            rgb = x[:, :, [2, 1, 0]]
            rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
            ax  = axes[count]
            ax[0].imshow(np.clip(rgb, 0, 1))
            ax[1].imshow(y, cmap="Greens", vmin=0, vmax=1)
            ax[2].imshow(p, cmap="Greens", vmin=0, vmax=1)
            iou = ((p > 0.5) & (y > 0.5)).sum() / (((p > 0.5) | (y > 0.5)).sum() + 1e-6)
            ax[2].set_title(f"IoU={iou:.3f}", fontsize=9)
            for a in ax: a.axis("off")
            count += 1
            if count >= n_samples: break
        if count >= n_samples: break

    plt.tight_layout()
    out = os.path.join(KAGGLE_OUTPUT, "predictions_preview.png")
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved: {out}")
    '''
    Mangrove_Labeled-0000000000-0000000000.tif
    Size     : 10496×10496px
    NDVI     : mean=0.245  range=[-0.20, 0.63]
    NDMI     : mean=0.075  range=[-0.33, 0.47]
    Mangrove : 31.0% of sampled 2048px window

  Mangrove_Labeled-0000000000-0000010496.tif
    Size     : 10496×10496px
    NDVI     : mean=0.167  range=[-0.20, 0.66]
    NDMI     : mean=0.132  range=[-0.33, 0.47]
    Mangrove : 24.1% of sampled 2048px window

  Mangrove_Labeled-0000000000-0000020992.tif
    Size     : 1273×10496px
    NDVI     : mean=0.399  range=[-0.24, 0.64]
    NDMI     : mean=0.188  range=[-0.33, 0.43]
    Mangrove : 89.7% of sampled 2048px window

  Mangrove_Labeled-0000010496-0000000000.tif
    Size     : 10496×10496px
    NDVI     : mean=0.001  range=[-0.26, 0.63]
    NDMI     : mean=0.108  range=[-0.28, 0.48]
    Mangrove : 22.1% of sampled 2048px window

=== END DIAGNOSTICS ===
'''
# ─────────────────────────────────────────────
# 14. TILED INFERENCE (shared by satellite + drone)
# ─────────────────────────────────────────────
def predict_tiled(model, x_full, mean, std, config):
    """Sliding-window inference with overlap averaging."""
    x_norm = (x_full - mean) / std
    h, w   = x_full.shape[:2]
    ps, st = config["patch_size"], config["stride"]
    acc    = np.zeros((h, w), dtype=np.float32)
    cnt    = np.zeros((h, w), dtype=np.float32)

    for i in range(0, max(h - ps + 1, 1), st):
        for j in range(0, max(w - ps + 1, 1), st):
            i2, j2 = min(i + ps, h), min(j + ps, w)
            patch  = x_norm[i:i2, j:j2, :]
            ph, pw = patch.shape[:2]
            if ph < ps or pw < ps:
                pad          = np.zeros((ps, ps, N_CHANNELS), dtype=np.float32)
                pad[:ph, :pw] = patch
                patch        = pad
            pred             = model.predict(patch[np.newaxis], verbose=0)[0, :, :, 0]
            acc[i:i2, j:j2] += pred[:i2-i, :j2-j]
            cnt[i:i2, j:j2] += 1

    return acc / np.maximum(cnt, 1)


# ─────────────────────────────────────────────
# 15. SATELLITE INFERENCE
# ─────────────────────────────────────────────
def infer_satellite(tif_path, model, mean, std, config, threshold=0.5):
    print(f"\nSatellite inference: {os.path.basename(tif_path)}")
    with rasterio.open(tif_path) as src:
        bands = [b + 1 for b in config["input_bands"]]
        data  = src.read(bands).astype(np.float32)
        data  = np.transpose(data, (1, 2, 0))
        meta  = src.meta.copy()

    prob_map = predict_tiled(model, data, mean, std, config)
    fname    = os.path.basename(tif_path).replace(".tif", "_mask.tif")
    out_path = os.path.join(KAGGLE_OUTPUT, fname)
    meta.update(count=1, dtype="float32", compress="lzw")
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(prob_map[np.newaxis])
    print(f"  Coverage : {100*(prob_map > threshold).mean():.1f}%  |  Saved: {out_path}")
    return prob_map


# ─────────────────────────────────────────────
# 16. DRONE INFERENCE
# ─────────────────────────────────────────────
def prepare_drone_frame(img_bgr):
    """Synthesise 8-channel input from a standard RGB drone frame."""
    img     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    b, g, r = img[:, :, 2], img[:, :, 1], img[:, :, 0]
    nir     = np.clip(r * 0.8 + g * 0.4, 0, 1)
    swir    = np.clip(r * 0.6, 0, 1)
    ndvi    = (nir - r)    / (nir + r    + 1e-6)
    ndwi    = (g   - nir)  / (g   + nir  + 1e-6)
    ndmi    = (nir - swir) / (nir + swir + 1e-6)
    # Order must match CONFIG["input_bands"]: B2,B3,B4,B8,B11,NDVI,NDWI,NDMI
    return np.stack([b, g, r, nir, swir, ndvi, ndwi, ndmi], axis=-1).astype(np.float32)


def overlay_mask(frame_rgb, prob_map, threshold=0.5, alpha=0.45):
    overlay  = frame_rgb.copy()
    mask_bin = (prob_map > threshold).astype(np.uint8)
    green    = np.zeros_like(frame_rgb); green[:, :, 1] = 255
    fg       = mask_bin.astype(bool)
    if fg.any():
        overlay[fg] = ((1 - alpha) * frame_rgb[fg] + alpha * green[fg]).astype(np.uint8)
    contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (0, 200, 0), 1)
    cov  = 100 * mask_bin.sum() / mask_bin.size
    conf = float(prob_map[fg].mean() * 100) if fg.any() else 0.0
    cv2.rectangle(overlay, (8, 8), (230, 72), (0, 0, 0), -1)
    cv2.putText(overlay, f"Mangrove: {cov:.1f}%",    (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 100), 1)
    cv2.putText(overlay, f"Confidence: {conf:.1f}%", (14, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 200, 200), 1)
    return overlay


def infer_drone_photo(image_path, model, mean, std, config, threshold=0.5):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")
    prob_map = predict_tiled(model, prepare_drone_frame(img_bgr), mean, std, config)
    overlay  = overlay_mask(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), prob_map, threshold)
    out = os.path.join(KAGGLE_OUTPUT,
                       os.path.basename(image_path).rsplit(".", 1)[0] + "_mangrove.jpg")
    cv2.imwrite(out, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    print(f"  Saved: {out}")
    return prob_map


def infer_drone_video(video_path, model, mean, std, config, threshold=0.5, every_n=3):
    cap       = cv2.VideoCapture(video_path)
    fps       = cap.get(cv2.CAP_PROP_FPS) or 25.0
    fw, fh    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total     = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out_path  = os.path.join(KAGGLE_OUTPUT,
                              os.path.basename(video_path).rsplit(".", 1)[0] + "_mangrove.mp4")
    writer    = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (fw, fh))
    last_prob = np.zeros((fh, fw), dtype=np.float32)

    for idx in range(total):
        ret, frame = cap.read()
        if not ret:
            break
        if idx % every_n == 0:
            last_prob = predict_tiled(model, prepare_drone_frame(frame), mean, std, config)
        writer.write(cv2.cvtColor(
            overlay_mask(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), last_prob, threshold),
            cv2.COLOR_RGB2BGR))
        if idx % 60 == 0:
            print(f"  Frame {idx}/{total}")

    cap.release(); writer.release()
    writer.release()
    tf.keras.backend.clear_session() # Resets and drops stale graph allocations
    gc.collect()
    print(f"  Saved: {out_path}")


class MemoryCleanupCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Force Python's garbage collector to free unreferenced array blocks
        gc.collect()
        # Clear backend graph memory allocations that linger between epochs
        #tf.keras.backend.clear_session()

# ─────────────────────────────────────────────
# 17. TRAIN
# ─────────────────────────────────────────────
def train():
    print("\n=== MANGROVE SEGMENTATION v6 — KAGGLE GPU ===\n")

    os.makedirs("/kaggle/temp", exist_ok=True)

    # Confirm NDVI/NDMI signal exists before committing to full training run
    diagnose_dataset(CONFIG)

    mean, std              = load_or_compute_stats(CONFIG)
    train_files, val_files = get_file_split(CONFIG)

    train_ds = make_dataset(train_files, mean, std, CONFIG,
                         augment=True, shuffle=True,
                         max_patches=CONFIG["max_train_patches"],
                         cache_path="/kaggle/temp/train_cache")

    val_ds = make_dataset(val_files, mean, std, CONFIG,
                       augment=False, shuffle=False,
                       max_patches=CONFIG["max_val_patches"],
                       cache_path="/kaggle/temp/val_cache")

    # Explicit step counts prevent math.log10(0) crash in Keras 3
    # when dataset size cannot be inferred from a generator.
    train_steps = max(1, CONFIG["max_train_patches"] // CONFIG["batch_size"])
    val_steps   = max(1, CONFIG["max_val_patches"]   // CONFIG["batch_size"])
    print(f"\nTrain steps/epoch : {train_steps}")
    print(f"Val   steps/epoch : {val_steps}")

    input_shape = (CONFIG["patch_size"], CONFIG["patch_size"], N_CHANNELS)

    if os.path.exists(MODEL_PATH):
        print(f"\nLoading existing model: {MODEL_PATH}")
        model = load_model(MODEL_PATH, custom_objects={
            "bce_dice_loss": bce_dice_loss,
            "dice_loss":     dice_loss,
            "IoUMetric":     IoUMetric,
        })
        print("\nEvaluating on validation set...")
        model.evaluate(val_ds, steps=val_steps)
    else:
        with strategy.scope():
            model = build_unet(input_shape)
            model.compile(
                optimizer=tf.keras.optimizers.Adam(1e-4, clipnorm=1.0),
                loss=bce_dice_loss,
                metrics=[IoUMetric()],
            )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_iou_metric", patience=7,
                restore_best_weights=True, mode="max", verbose=1),
            tf.keras.callbacks.ModelCheckpoint(
                MODEL_PATH, monitor="val_iou_metric",
                save_best_only=True, mode="max", verbose=1),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_iou_metric", factor=0.5,
                patience=3, min_lr=1e-6, mode="max", verbose=1),
            tf.keras.callbacks.CSVLogger(
                os.path.join(KAGGLE_OUTPUT, "training_log.csv")),
            MemoryCleanupCallback()
        ]

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=CONFIG["epochs"],
            steps_per_epoch=train_steps,
            validation_steps=val_steps,
            callbacks=callbacks,
        )
        plot_training_history(history)
        visualize_predictions(model, val_ds)
        print(f"\nModel saved: {MODEL_PATH}")

    return model, mean, std


In [ ]:
import os
print("Verifying dataset access...")
folder = CONFIG["folder"]
print("Folder exists:", os.path.exists(folder))
tif_count = len([f for f in os.listdir(folder) if f.lower().endswith('.tif')])
print(f"Direct .tif files: {tif_count}")

# List first few
print("Sample files:")
for f in sorted(os.listdir(folder))[:5]:
    if f.lower().endswith('.tif'):
        print("   ", f)


# Verify what your 10 bands actually are
import rasterio, os
path = os.path.join(CONFIG["folder"], os.listdir(CONFIG["folder"])[0])
with rasterio.open(path) as src:
    print(src.descriptions)   # prints band names if they're stored in the TIF
    print(src.count, "bands")


import os

print("=== FULL INPUT STRUCTURE ===")
base = "/kaggle/input"
for root, dirs, files in os.walk(base):
    tif_count = sum(1 for f in files if f.lower().endswith('.tif'))
    if tif_count > 0 or any(d.startswith('sundarban') for d in dirs) or 'sundarban' in root.lower():
        print(f"→ {root} | TIFs: {tif_count} | Dirs: {dirs[:5]}")
        if tif_count > 0:
            print("   Sample files:", [f for f in files if f.lower().endswith('.tif')][:3])


In [ ]:
# ─────────────────────────────────────────────
# RUN TRAINING
# ─────────────────────────────────────────────
model, mean, std = train()


In [ ]:
# ─────────────────────────────────────────────
# OPTIONAL: INFERENCE (run after training, or after loading a saved model)
# ─────────────────────────────────────────────
# Uncomment and set a path to run inference with the trained model:

# infer_satellite("/path/to/scene.tif", model, mean, std, CONFIG)
# infer_drone_photo("/path/to/photo.jpg", model, mean, std, CONFIG)
# infer_drone_video("/path/to/video.mp4", model, mean, std, CONFIG, every_n=3)


## Notes

```
═══════════════════════════════════════════════════════
KAGGLE NOTEBOOK — paste entire file as one cell, then:

  model, mean, std = train()

If diagnose_dataset() shows 0% mangrove coverage, lower thresholds:
  CONFIG["ndvi_low"] = 0.20   (try 0.20 → 0.15 progressively)
  CONFIG["ndmi_low"] = 0.00   (remove moisture gate entirely)

All outputs → /kaggle/working/  (visible in Output tab)
═══════════════════════════════════════════════════════
```
